- https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/quantum-phase-estimation#33-exercise

In [ ]:
import math
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Sampler

## Create a QPE circuit

The $T$ gate can be expressed as follows:
$$
T = \begin{bmatrix}
1 & 0\\ 
0 & e^{i\frac{\pi}{4}}
\end{bmatrix}
$$

The $T$ gate acts on $\ket{1}$ as:
 
$$
T|1⟩ = e^{i\pi/4} \ket{1}
$$

 
We use $\ket{v_{\theta}} = \ket{1}$ as the eigenvector (prepared with an $X$ gate) and 3 counting qubits, so $L$ = 3.
 
In the controlled-U loop, the gate `cp(π/4, n)` *is* the controlled-$T$: a controlled-phase of $\frac{\pi}{4}$ applied to qubit $n$, with the repetition count doubling each counting qubit to realize the `U^(2^j)` powers.

In [ ]:
# the number of counting qubits
L = 3

In [ ]:
qc = QuantumCircuit(L + 1, L)

# Prepare the eigenstate \ket{v_{\theta}} = |1> on qubit 3
qc.x(L)
qc.barrier()

# Step 2: put the L counting qubits into superposition
for qubit in range(L):
    qc.h(qubit)
qc.barrier()

# Step 3: apply controlled-U^(2^j) operations (here U = T, so cp(pi/4) is controlled-T)

for j in range(L):

    # Method 1
    for i in range(2**j):
        qc.cp(math.pi / 4, j, L)  # This is C-U
    
    # Method 2
    #qc.cp((2**j) * math.pi / 4, j, L)  # This is C-U

    
qc.barrier()

# Step 4: apply inverse QFT on the counting register
qc.append(
    instruction=QFTGate(num_qubits=L).inverse(annotated=True),
    qargs=range(L),
)
qc.barrier()

# Step 5: measure the counting register
for i in range(L):
    qc.measure(i, i)

In [ ]:
qc.draw(output="mpl")

## Transpile and simulate

In [ ]:
simulator = AerSimulator()

pass_manager = generate_preset_pass_manager(backend=simulator, optimization_level=1)

tqc = pass_manager.run(qc)

In [ ]:
tqc.draw(output='mpl')

In [ ]:
sampler = Sampler(mode=simulator)
job = sampler.run([tqc], shots=2048)
result = job.result()

In [ ]:
counts = result[0].data.c.get_counts()

In [ ]:
plot_histogram(counts)

## Postprocessing

In [ ]:
j = max(counts, key=counts.get)

In [ ]:
j = int(j, base=2)
print(f'{j=}')

$$ \Large \theta = \frac{2\pi j^{\prime}}{2^{L}}  $$

In [ ]:
theta_estim = (2*math.pi*j) / (2**L)
print(f'{theta_estim=}')

In [ ]:
theta_true = math.pi / 4

In [ ]:
math.isclose(theta_estim, theta_true)